In [ ]:
import requests
def get_gpt_response(query):
    conversation_history.append({"role": "user", "content": query})
    
    data = {
        "model": "gpt-4",
        "messages": conversation_history,
        "temperature": 0
    }
    
    try:
        response = requests.post(url, headers=headers, data=json.dumps(data))
        response_json = response.json()
        
        if "choices" in response_json:
            answer = response_json["choices"][0]["message"]["content"]
            conversation_history.append({"role": "assistant", "content": answer})
            print("> Answer: ", answer)
            return answer
        else:
            print("⚠️ API 响应异常:", response_json)
            return "API_ERROR"
    
    except requests.exceptions.RequestException as e:
        print(f"⚠️ 请求异常: {e}")
        return "REQUEST_ERROR"
    except json.JSONDecodeError:
        print("⚠️ JSON 解析错误")
        return "JSON_ERROR"



url = ""
headers = { 
    "Content-Type": "", 
    "Authorization": ""
}

# 存储对话历史
conversation_history = [
    {"role": "system", "content": "You are a useful helper."}
]

def add_context(text):
    conversation_history.append({"role": "user", "content": text})

In [3]:
import json
with open("qa_list.json", "r", encoding="utf-8") as f:
    golden_answers = json.load(f)

In [4]:
with open("rr_qa_answers_ours_claude.json", "r", encoding="utf-8") as f:
    ours_answers = json.load(f)

In [ ]:
instruction = 'We would like to request your feedback on the performance of the AI assistant in response to the user question displayed above according to the gold answer. Please use the following listed aspects and their descriptions as evaluation criteria: \n - Accuracy and Hallucinations: The assistant’s answer is semantically consistent with the gold answer; The numerical value and order need to be accurate, and there should be no hallucinations. \n- Completeness: Referring to the reference answers, the assistant’s answer should contain all the key points needed to answer the user’s question; further elaboration on these key points can be omitted. Please rate whether this answer is suitable for the question. Please note that the gold answer can be considered as a correct answer to the question. The assistant receives an overall score on a scale of 1 to 100, where a higher score indicates better overall performance.Please note that if the assistant’s answer and the gold answer fully meet the above criteria, its overall rating should be the full marks (100). Please first provide a comprehensive explanation of your evaluation, avoiding any potential bias.Then, output a line indicating the score of the Assistant. PLEASE OUTPUT WITH THE FOLLOWING FORMAT, WHERE THE SCORE IS A SCALE OF 1 TO 100 BY STRICTLY FOLLOWING THIS FORMAT: "[[score]]", FOR EXAMPLE "Rating: [[100]]": \n <Start Output>\nEvaluation evidence: your evluation explanation here, no more than 100 words Rating: [[score]]\n<End Output>\nNow, start your evaluation:'
score = []

for idx in range(0, len(golden_answers)):
    conversation_history.clear()
    qa_lst = golden_answers[idx]["qa_output"]
    for pair_idx in range(len(qa_lst)):
        qa_pair = qa_lst[pair_idx]
        question = qa_pair["question"]
        gold_answer = qa_pair["answer"]
        add_context("[The given question]: " + question)
        add_context("[Gold Answer]: " + gold_answer)
        llm_predict = ours_answers[idx]["answers"][pair_idx]["answer"]
        add_context("[The start of Assistant's predicted Answer]\n" + llm_predict + "[The End of Assistant’s Predicted Answer]")
        response = get_gpt_response(instruction)
        score.append(response)

> Answer:  <Start Output>
Evaluation evidence: The assistant's answer diverges significantly from the gold answer, which states there is no significant impact on the public prosecution service. The assistant introduces several points not mentioned in the gold answer, such as specialized training and differences in prosecution rates, which are not supported by the gold answer. This indicates hallucinations and a lack of accuracy. The assistant's response also lacks completeness as it does not align with the gold answer's key point of no significant impact. Therefore, the assistant's performance is rated low due to these discrepancies.
Rating: [[30]]
<End Output>
> Answer:  <Start Output>
Evaluation evidence: The assistant's answer lacks specificity and does not accurately reflect the gold answer. The gold answer specifies the use of a case management system with specific flags for different case types, which is a crucial detail missing from the assistant's response. The assistant's answ

In [ ]:
# parse for the legal answer
import re
def extract_numbers(text):
    """提取文本中的所有数字"""
    numbers = re.findall(r'\d+', text)
    return [int(num) for num in numbers]

score_num = []
idx = 0
for output in score:
    if output == "API_ERROR" or output == "REQUEST_ERROR":
        score_num.append(0)
    else:
        score_num.append(extract_numbers(output)[-1])

In [ ]:
def average(lst):
    return sum(lst) / len(lst) if lst else 0

In [ ]:
average(score_num)